# YOLO26m-seg Tuning & Evaluation Pipeline

This notebook provides a simplified, modular pipeline to train and evaluate **YOLO26m-seg** on leaf disease segmentation datasets (Rice or Coffee).

References:
- [YOLO26](https://docs.ultralytics.com/models/yolo26)
- [Ultralytics Segmentation](https://docs.ultralytics.com/tasks/segment)
- [HuggingFace Upload](https://huggingface.co/docs/huggingface_hub/en/guides/upload)
- [ONNX Export for YOLO26 Models](https://docs.ultralytics.com/integrations/onnx)

## 1. Setup, Environment, and Configuration

In [ ]:
import json
import os
import shutil
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
import torch
import yaml

sys.path.append(os.path.abspath("."))

from hf_upload import upload_folder_to_hf
from yolo26seg import benchmark_cpu_latency, evaluate_test_set, run_train
from yolo_coverter import build_coco_index, convert_to_yolo, generate_splits

from utils import set_seed


# Global seed
SEED = 3407
set_seed(SEED)

# Set target domain: "coffee" or "rice"
TARGET_DOMAIN = "coffee"
assert TARGET_DOMAIN in {"rice", "coffee"}

# Directory setup
PROJECT_ROOT = Path(".").resolve().parent
RAW_ROOT = Path("datasets").resolve()
WORK_DIR = Path(f"./yolo26_seg_{TARGET_DOMAIN}").resolve()

DATASET_DIR = WORK_DIR / "dataset"
RUNS_DIR = WORK_DIR / "runs"
ARTIFACTS_DIR = WORK_DIR / "artifacts"

for directory in [DATASET_DIR, RUNS_DIR, ARTIFACTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Environment setup for YOLO config directory
os.environ["YOLO_CONFIG_DIR"] = str(WORK_DIR / "config")

# Domain categories mapping
RICE_CLASSES = {"Healthy", "BrownSpot", "Hispa", "LeafBlast"}
COFFEE_FOLDER_TO_CLASS = {"0": "LeafMiner", "1": "PowderyMildew", "2": "Rust", "3": "AlgalLeafSpot"}

DOMAIN_CLASSES = sorted(RICE_CLASSES if TARGET_DOMAIN == "rice" else set(COFFEE_FOLDER_TO_CLASS.values()))
DOMAIN_CLASS_TO_ID = {name: idx for idx, name in enumerate(DOMAIN_CLASSES)}
DEVICE = "0" if torch.cuda.is_available() else "cpu"

print(f"Target domain: {TARGET_DOMAIN}")
print(f"Classes: {DOMAIN_CLASSES}")
print(f"Local YOLO class map: {DOMAIN_CLASS_TO_ID}")
print(f"Device: {DEVICE}")
print(f"Work directory: {WORK_DIR}")

## 2. Load dataset annotations and generate train/val/test splits

In [ ]:
COCO_PATH = RAW_ROOT / f"{TARGET_DOMAIN}_leaf_disease" / "annotations.coco.json"

if not COCO_PATH.exists():
    raise FileNotFoundError(f"COCO annotations file not found at: {COCO_PATH}")

with COCO_PATH.open("r", encoding="utf-8") as f:
    coco_data = json.load(f)

print(f"Loaded {TARGET_DOMAIN.title()} COCO dataset.")
print(f"Images: {len(coco_data['images'])}, Annotations: {len(coco_data['annotations'])}")

# Generate train/val/test splits (70% / 15% / 15%)
split_df = generate_splits(
    coco_data=coco_data, target_domain=TARGET_DOMAIN, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=SEED
)

print("\nSplits summary per class:")
print(split_df.groupby(["split", "label"]).size())

## 3. Convert COCO polygons to YOLO format

In [ ]:
coco_index = build_coco_index(coco_data, TARGET_DOMAIN)
converted_df = convert_to_yolo(
    df=split_df,
    target_domain=TARGET_DOMAIN,
    raw_root=RAW_ROOT,
    dataset_dir=DATASET_DIR,
    artifacts_dir=ARTIFACTS_DIR,
    coco_index=coco_index,
    domain_class_to_id=DOMAIN_CLASS_TO_ID,
)

print(f"\nSuccessfully converted {len(converted_df)} images to YOLO format.")
print(f"Skipped images: {len(split_df) - len(converted_df)}")

# Generate data.yaml config file
DATA_YAML = WORK_DIR / f"data_{TARGET_DOMAIN}.yaml"
data_yaml_content = {
    "path": str(DATASET_DIR.resolve()).replace("\\", "/"),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {idx: name for idx, name in enumerate(DOMAIN_CLASSES)},
    "nc": len(DOMAIN_CLASSES),
}
with DATA_YAML.open("w", encoding="utf-8") as f:
    yaml.safe_dump(data_yaml_content, f, sort_keys=False)

print(f"Written dataset configuration to: {DATA_YAML}")

## 4. Visualize converted Ground-Truth overlays

In [ ]:
sample_df = converted_df.groupby("split", group_keys=False).head(2)
fig, axes = plt.subplots(1, len(sample_df), figsize=(4 * len(sample_df), 4))
if len(sample_df) == 1:
    axes = [axes]
for ax, (_, row) in zip(axes, sample_df.iterrows()):
    img_overlay = overlay_segments(Path(row["image_path_yolo"]), Path(row["label_path"]))
    ax.imshow(img_overlay)
    ax.set_title(f"{row['split']} | {row['label']}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Train the YOLO26m-seg Model

In [ ]:
YOLO_MODEL = os.environ.get("YOLO_MODEL", "yolo26m-seg.pt")
EPOCHS = int(os.environ.get("YOLO_EPOCHS", "50"))
BATCH = int(os.environ.get("YOLO_BATCH", "8"))

COMMON_ARGS = {
    "imgsz": 640,
    "batch": BATCH,
    "workers": 2,
    "seed": SEED,
    "deterministic": True,
    "project": str(RUNS_DIR),
    "exist_ok": True,
    "plots": True,
    "save": True,
    "val": True,
}

train_config = {
    "name": f"yolo26m_seg_{TARGET_DOMAIN}",
    "model": YOLO_MODEL,
    "epochs": EPOCHS,
    "optimizer": "auto",
    "patience": 15,
}

train_metrics = run_train(train_config, COMMON_ARGS, DATA_YAML, DEVICE)
print("\nTraining complete.")
print(train_metrics)

## 6. Calculate Metrics on Test Set

Computes mIoU, IoU per class, Dice Score, mAP@50, mAP@50:95, and CPU latency (ms).

In [ ]:
best_ckpt = Path(train_metrics["best_ckpt"])
print(f"Best checkpoint: {best_ckpt}")

# Calculate semantic overlap metrics & YOLO mAP metrics on test set split
custom_summary, score_details_df = evaluate_test_set(
    ckpt=best_ckpt,
    df=converted_df,
    domain_classes=DOMAIN_CLASSES,
    data_yaml=DATA_YAML,
    imgsz=640,
    device=DEVICE,
)

# Calculate CPU latency benchmark
print("\nBenchmarking inference CPU latency...")
cpu_latency_ms = benchmark_cpu_latency(
    best_ckpt,
    converted_df,
    imgsz=640,
    n_images=50,
)

# Combine all metrics
final_results = {
    "dataset": TARGET_DOMAIN,
    "model": YOLO_MODEL,
    "mAP50_mask": custom_summary["mAP50_mask"],
    "mAP50_95_mask": custom_summary["mAP50_95_mask"],
    "mIoU": custom_summary["mIoU"],
    "Dice Score": custom_summary["Dice"],
    "cpu_latency_ms": cpu_latency_ms,
}

for cls in DOMAIN_CLASSES:
    final_results[f"{cls}_IoU"] = custom_summary[f"{cls}_IoU"]
    final_results[f"{cls}_Dice"] = custom_summary[f"{cls}_Dice"]

print("\nFinal Evaluation Metrics:")
for k, v in final_results.items():
    if isinstance(v, float):
        print(f" - {k}: {v:.6f}")
    else:
        print(f" - {k}: {v}")

# Save metric files to disk
score_details_df.to_csv(ARTIFACTS_DIR / f"yolo26_seg_{TARGET_DOMAIN}_custom_mask_scores.csv", index=False)
summary_df = pd.DataFrame([final_results])
summary_df.to_csv(ARTIFACTS_DIR / f"yolo26_seg_{TARGET_DOMAIN}_summary.csv", index=False)
print(f"\nSaved custom scores and summary metrics to: {ARTIFACTS_DIR}")

## 7. Save Artifacts and Generate Model Card

In [ ]:
shutil.copy2(best_ckpt, ARTIFACTS_DIR / f"best_yolo26_seg_{TARGET_DOMAIN}.pt")
shutil.copy2(DATA_YAML, ARTIFACTS_DIR / f"data_{TARGET_DOMAIN}.yaml")
(ARTIFACTS_DIR / "class_names.json").write_text(json.dumps(DOMAIN_CLASSES, indent=2), encoding="utf-8")

# Create Model README
model_card = f"""# YOLO26-seg Phase 3 - {TARGET_DOMAIN.title()} Leaf Disease Segmentation

## Task
Instance segmentation for Vietnamese {TARGET_DOMAIN} leaf disease.

## Classes
{DOMAIN_CLASSES}

## Metrics
- mAP@50 (mask): {final_results["mAP50_mask"]:.4f}
- mAP@50:95 (mask): {final_results["mAP50_95_mask"]:.4f}
- mIoU: {final_results["mIoU"]:.4f}
- Dice Score: {final_results["Dice Score"]:.4f}
- CPU Latency: {final_results["cpu_latency_ms"]:.2f} ms/image

### Per-Class Metrics
"""
for cls in DOMAIN_CLASSES:
    model_card += f"- {cls}: IoU = {final_results[f'{cls}_IoU']:.4f}, Dice = {final_results[f'{cls}_Dice']:.4f}\n"

(ARTIFACTS_DIR / "README.md").write_text(model_card, encoding="utf-8")
print(f"Artifacts saved to: {ARTIFACTS_DIR}")
print(sorted(p.name for p in ARTIFACTS_DIR.iterdir()))

## 8. Push Artifacts to Hugging Face Hub

In [ ]:
HF_REPO_ID = os.environ.get("HF_REPO_ID", f"<team-or-user>/ml-vietnam-plant-disease-yolo26-seg-{TARGET_DOMAIN}")
HF_UPLOAD = os.environ.get("HF_UPLOAD", "0") == "1"

if HF_UPLOAD:
    print(f"Uploading folder to HF Repo: {HF_REPO_ID}")
    upload_folder_to_hf(HF_REPO_ID, ARTIFACTS_DIR, TARGET_DOMAIN)
else:
    print("HF_UPLOAD=0. Skipping Hugging Face upload.")
    print("To upload, set environment variables: HF_UPLOAD=1, HF_REPO_ID, and HF_TOKEN.")